In [1]:
import os
import logging
from typing import Any
from typing_extensions import TypedDict

from dotenv import load_dotenv
import chromadb
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

In [2]:
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)
logger = logging.getLogger(__name__)

logger.info("GROQ_API_KEY set: %s", bool(os.getenv("GROQ_API_KEY")))
logger.info("GOOGLE_API_KEY set: %s", bool(os.getenv("GOOGLE_API_KEY")))
logger.info("LLM_MODEL: %s", os.getenv("LLM_MODEL"))

2026-05-19 03:38:32,922 INFO GROQ_API_KEY set: True
2026-05-19 03:38:32,923 INFO GOOGLE_API_KEY set: True
2026-05-19 03:38:32,923 INFO LLM_MODEL: llama-3.3-70b-versatile


In [3]:
class VideoAnalysisState(TypedDict):
    question: str
    chat_history: list[Any]
    video_a_id: str
    video_b_id: str
    context_a: str
    context_b: str
    sources: list[dict]
    answer: str

In [4]:
def get_collection() -> chromadb.Collection:
    """Return the persistent ChromaDB collection for video chunks."""
    chroma_path = os.getenv("CHROMA_PATH", "./backend/chroma_db")
    client = chromadb.PersistentClient(path=chroma_path)
    return client.get_collection("video_chunks")

In [5]:
def retrieve_context(video_id: str, query: str, k: int = 4) -> tuple[str, list[dict]]:
    """Search ChromaDB for relevant chunks from a specific video.

    Returns the formatted context string and a list of source metadata dicts.
    """
    embeddings_model = GoogleGenerativeAIEmbeddings(model=os.getenv("EMBEDDING_MODEL"))
    query_embedding = embeddings_model.embed_query(query)

    collection = get_collection()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        where={"video_id": video_id},
        include=["documents", "metadatas", "distances"],
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    distances = results["distances"][0]

    if not docs:
        logger.warning("No chunks found for video_id=%s", video_id)
        return "", []

    # Prepend video-level metadata so the LLM sees title and engagement rate
    first = metas[0]
    header = (
        f"Video: {first['title']} by {first['creator']}\n"
        f"Engagement Rate: {first['engagement_rate']}\n\n"
    )
    context = header + "\n\n".join(docs)

    sources = [
        {
            "video_id": m["video_id"],
            "title": m["title"],
            "chunk_index": m["chunk_index"],
            "distance": round(d, 4),
        }
        for m, d in zip(metas, distances)
    ]

    logger.info("Retrieved %d chunks for video_id=%s", len(docs), video_id)
    return context, sources

In [6]:
def retrieve_video_a(state: VideoAnalysisState) -> dict:
    """LangGraph node: fetch context for video A."""
    context, sources = retrieve_context(state["video_a_id"], state["question"])
    return {"context_a": context, "sources": sources}

In [7]:
def retrieve_video_b(state: VideoAnalysisState) -> dict:
    """LangGraph node: fetch context for video B and merge sources."""
    context, sources = retrieve_context(state["video_b_id"], state["question"])
    return {"context_b": context, "sources": state.get("sources", []) + sources}

In [8]:
def get_llm() -> ChatGroq:
    """Return a configured ChatGroq instance."""
    return ChatGroq(
        model=os.getenv("LLM_MODEL"),
        temperature=0,
        streaming=False,
    )

In [9]:
def generate_answer(state: VideoAnalysisState) -> dict:
    """LangGraph node: compare both videos and produce the final answer."""
    llm = get_llm()

    system_content = (
        "You are a social media analyst comparing two YouTube videos.\n"
        "Use only the context provided below to answer the question.\n"
        "Be specific and cite evidence from the transcripts.\n\n"
        f"--- Video A ---\n{state['context_a']}\n\n"
        f"--- Video B ---\n{state['context_b']}"
    )

    messages = [SystemMessage(content=system_content)]
    messages.extend(state["chat_history"])
    messages.append(HumanMessage(content=state["question"]))

    response = llm.invoke(messages)
    logger.info("Answer generated (%d chars)", len(response.content))
    return {"answer": response.content}

In [10]:
graph = StateGraph(VideoAnalysisState)

graph.add_node("retrieve_video_a", retrieve_video_a)
graph.add_node("retrieve_video_b", retrieve_video_b)
graph.add_node("generate_answer", generate_answer)

graph.set_entry_point("retrieve_video_a")
graph.add_edge("retrieve_video_a", "retrieve_video_b")
graph.add_edge("retrieve_video_b", "generate_answer")
graph.add_edge("generate_answer", END)

app = graph.compile()
print("Graph compiled successfully")

Graph compiled successfully


In [11]:
# SVTPv4sI_Jc: "The CIA's Worst New Tech Idea?" by Veritasium (engagement_rate=3.3747)
# B3m3AMRlYfc: "My first science video in 3 years" by Physics Girl (engagement_rate=17.1581)

result = app.invoke({
    "question": "Which video has better engagement and why?",
    "chat_history": [],
    "video_a_id": "SVTPv4sI_Jc",
    "video_b_id": "B3m3AMRlYfc",
    "context_a": "",
    "context_b": "",
    "sources": [],
    "answer": "",
})

print(result["answer"])
print("\nSources retrieved:")
for s in result["sources"]:
    print(f"  [{s['video_id']}] {s['title']} — chunk {s['chunk_index']} (dist={s['distance']})")

2026-05-19 03:38:52,085 INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2026-05-19 03:38:52,242 INFO Retrieved 4 chunks for video_id=SVTPv4sI_Jc
2026-05-19 03:38:53,219 INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2026-05-19 03:38:53,259 INFO Retrieved 4 chunks for video_id=B3m3AMRlYfc
2026-05-19 03:38:54,675 INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19 03:38:54,709 INFO Answer generated (1343 chars)


Video B, "My first science video in 3 years" by Physics Girl, has a significantly better engagement rate of 17.1581 compared to Video A, "The CIA's Worst New Tech Idea?" by Veritasium, which has an engagement rate of 3.3747.

The reason for this difference in engagement can be inferred from the tone and style of the two videos. Video B is more conversational and personal, with the host, Physics Girl, sharing her excitement about creating her first science video in 3 years and discussing her experience with long COVID. The video also has a more relaxed and casual tone, with the host filming from her bed and interacting with someone off-camera. This informal and personal approach may have helped to create a sense of connection with the audience, leading to higher engagement.

In contrast, Video A is more formal and informative, with the host, Veritasium, presenting a detailed explanation of a complex scientific topic. While the video is well-produced and engaging, it may not have resonat

In [12]:
follow_up = app.invoke({
    "question": "What specific topics in the higher-engagement video likely drove that result?",
    "chat_history": [
        HumanMessage(content="Which video has better engagement and why?"),
        AIMessage(content=result["answer"]),
    ],
    "video_a_id": "SVTPv4sI_Jc",
    "video_b_id": "B3m3AMRlYfc",
    "context_a": "",
    "context_b": "",
    "sources": [],
    "answer": "",
})

print(follow_up["answer"])

2026-05-19 03:39:22,901 INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2026-05-19 03:39:22,929 INFO Retrieved 4 chunks for video_id=SVTPv4sI_Jc
2026-05-19 03:39:23,916 INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2026-05-19 03:39:23,957 INFO Retrieved 4 chunks for video_id=B3m3AMRlYfc
2026-05-19 03:39:25,940 INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19 03:39:25,942 INFO Answer generated (2607 chars)


In the higher-engagement video, "My first science video in 3 years" by Physics Girl, several specific topics and elements likely contributed to the higher engagement rate. Some of these topics include:

1. **Neutrinos**: The video's focus on neutrinos, a fascinating and mysterious topic in physics, may have piqued the audience's interest. The host's enthusiasm and curiosity about neutrinos, as evident in statements like "This is a cool topic" and "They're kind of like your uncle as a ghost. Like kind of tapping your shoulder and being like, 'There's still more to learn about the universe,'" may have been infectious and engaging for the audience.
2. **Personal story**: The host's personal story of returning to creating science videos after a 3-year hiatus, and her experience with long COVID, may have created an emotional connection with the audience. This personal touch, as seen in statements like "We're working from the bed Yeah. as you have to with long COVID," can make the content mo